In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


Cell 1 — Load df, verify gender columns, fix LDA merge only if actually needed

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import pearsonr
import os

pd.set_option('display.max_columns', None)

BASE = '/content/drive/MyDrive/Transcripts_CSS/outputs'
DF_PATH = f'{BASE}/df-all-features.csv'

df = pd.read_csv(DF_PATH)
print(f"Loaded df-all-features.csv: {df.shape}")

# verify gender columns already present and clean (should be, from earlier pipeline)
gender_cols_present = all(c in df.columns for c in ['male_seconds', 'female_seconds', 'dominant_gender'])
print(f"Gender columns present: {gender_cols_present}")
print(f"Nulls in male_seconds: {df['male_seconds'].isna().sum()}")
print(f"dominant_gender value counts:\n{df['dominant_gender'].value_counts()}")

# verify topic columns already present
topic_cols_present = [c for c in df.columns if c.endswith('_Probability')]
print(f"\nTopic probability columns already in df: {len(topic_cols_present)}")

Loaded df-all-features.csv: (8847, 178)
Gender columns present: True
Nulls in male_seconds: 0
dominant_gender value counts:
dominant_gender
male      5232
female    3615
Name: count, dtype: int64

Topic probability columns already in df: 80


Cell 2 — Corpus-wide (non-topic) correlations against gender

In [ ]:
COMMUNITY_SHORT_NAME = {
    "transcripts_business": "business", "transcripts_religion": "religion",
    "transcripts_comedy": "comedy", "transcripts_lifestyle": "lifestyle",
    "transcripts_tech": "tech", "politics_transcripts": "politics",
    "gaming_transcripts": "gaming", "motivational_transcripts": "motivational",
}

df_gdcf = df.dropna(subset=['male_seconds', 'female_seconds']).copy()
print(f"Rows used for GDCF: {len(df_gdcf)}")

# non-topic features only — no fillna(0) needed here since these are corpus-wide, real values
non_topic_cols = [c for c in df_gdcf.select_dtypes(include='number').columns
                   if not c.endswith('_Probability') and c not in ['male_seconds', 'female_seconds']]

results = []
for feat in non_topic_cols:
    col = df_gdcf[feat]
    if col.std() == 0 or col.isna().all():
        continue
    for gender in ['male_seconds', 'female_seconds']:
        valid = pd.concat([col, df_gdcf[gender]], axis=1).dropna()
        if len(valid) < 10 or valid.iloc[:, 0].std() == 0 or valid.iloc[:, 1].std() == 0:
            continue
        corr, pval = pearsonr(valid.iloc[:, 0], valid.iloc[:, 1])
        results.append({'feature': feat, 'gender': gender, 'correlation': round(corr, 4),
                         'pvalue': pval, 'n': len(valid)})

corpus_results_df = pd.DataFrame(results)
n_tests = len(corpus_results_df)
alpha_bonferroni = 0.05 / n_tests
print(f"Corpus-wide tests: {n_tests}, Bonferroni alpha: {alpha_bonferroni:.2e}")

sig_corpus = corpus_results_df[
    (corpus_results_df['pvalue'] < alpha_bonferroni) & (corpus_results_df['correlation'].abs() >= 0.1)
].copy()
sig_corpus = sig_corpus.reindex(sig_corpus['correlation'].abs().sort_values(ascending=False).index)
print(f"Significant corpus-wide correlations: {len(sig_corpus)}")
display(sig_corpus.head(30))

Rows used for GDCF: 8847
Corpus-wide tests: 166, Bonferroni alpha: 3.01e-04
Significant corpus-wide correlations: 45


,feature,gender,correlation,pvalue,n
135,deprel_cc_count,female_seconds,0.2180,1.124727e-95,8847
149,deprel_nmod:poss_count,female_seconds,0.2178,1.765656e-95,8847
19,upos_CCONJ_count,female_seconds,0.2115,4.836686e-90,8847
85,deprel_xcomp_count,female_seconds,0.2041,8.513070e-84,8847
41,upos_ADV_count,female_seconds,0.1956,5.041497e-77,8847
55,deprel_advmod_count,female_seconds,0.1921,2.463640e-74,8847
25,upos_PRON_count,female_seconds,0.1902,7.968295e-73,8847
99,deprel_mark_count,female_seconds,0.1852,4.444225e-69,8847
95,deprel_ccomp_count,female_seconds,0.1796,5.369313e-65,8847
29,upos_DET_count,female_seconds,0.1791,1.221085e-64,8847


Cell 3 — Per-community topic correlations (no fillna, subset-only, no cross-community leakage)

In [ ]:
LDA_DIR = f'{BASE}/LDA_results'

def get_top_10_words(words_df, topic_num):
    row = words_df.iloc[topic_num - 1]
    return ", ".join(str(w) for w in row.iloc[1:11].tolist())

all_topic_results = []

for community, short_name in COMMUNITY_SHORT_NAME.items():
    community_df = df_gdcf[df_gdcf['community'] == community].copy()
    topic_cols = [c for c in community_df.columns if c.startswith(f'{short_name}_Topic_')]

    if not topic_cols:
        print(f"{community}: no topic columns found, skipping")
        continue

    words_path = f'{LDA_DIR}/LDA_topics_transcripts_{short_name}.csv'
    words_df = pd.read_csv(words_path, header=None) if os.path.exists(words_path) else None

    community_results = []
    for feat in topic_cols:
        col = community_df[feat]  # NOT filled with 0 — real values only, within this community's own rows
        if col.std() == 0 or col.isna().all():
            continue
        for gender in ['male_seconds', 'female_seconds']:
            valid = pd.concat([col, community_df[gender]], axis=1).dropna()
            if len(valid) < 10 or valid.iloc[:, 0].std() == 0 or valid.iloc[:, 1].std() == 0:
                continue
            corr, pval = pearsonr(valid.iloc[:, 0], valid.iloc[:, 1])
            community_results.append({'feature': feat, 'gender': gender, 'correlation': round(corr, 4),
                                       'pvalue': pval, 'n': len(valid), 'community': community})

    if not community_results:
        print(f"{community}: no valid correlations computed")
        continue

    results_df = pd.DataFrame(community_results)
    n_tests = len(results_df)
    alpha_adj = 0.05 / n_tests
    sig = results_df[(results_df['pvalue'] < alpha_adj) & (results_df['correlation'].abs() >= 0.1)].copy()

    if words_df is not None and len(sig) > 0:
        def lookup_words(feat):
            topic_num = int(feat.replace(f'{short_name}_Topic_', '').replace('_Probability', ''))
            try:
                return f"T{topic_num}: {get_top_10_words(words_df, topic_num)}"
            except IndexError:
                return ""
        sig['topic_words'] = sig['feature'].apply(lookup_words)

    print(f"{community}: {n_tests} tests, {len(sig)} significant")
    all_topic_results.append(sig)

topic_results_df = pd.concat(all_topic_results, ignore_index=True) if all_topic_results else pd.DataFrame()
topic_results_df.to_csv(f'{BASE}/gdcf_all_communities_topic_correlations.csv', index=False)
display(topic_results_df.sort_values('correlation', key=abs, ascending=False).head(30))

transcripts_business: 10 tests, 6 significant
transcripts_religion: 30 tests, 8 significant
transcripts_comedy: 10 tests, 3 significant
transcripts_lifestyle: 10 tests, 6 significant
transcripts_tech: 60 tests, 11 significant
politics_transcripts: no valid correlations computed
gaming_transcripts: no valid correlations computed
motivational_transcripts: no valid correlations computed


,feature,gender,correlation,pvalue,n,community,topic_words
19,lifestyle_Topic_2_Probability,male_seconds,0.3849,1.387110e-40,1112,transcripts_lifestyle,"T2: आज, आपक, लग, अच, चल, कर, हम, रह, बन, इतन"
8,religion_Topic_3_Probability,male_seconds,0.3553,1.220402e-27,882,transcripts_religion,"T3: jesus, god, new, pray, today, christ, prop..."
5,business_Topic_5_Probability,female_seconds,0.3518,9.289593e-30,974,transcripts_business,"T5: okay, going, one, see, right, oil, war, ye..."
4,business_Topic_5_Probability,male_seconds,-0.3472,5.551675e-29,974,transcripts_business,"T5: okay, going, one, see, right, oil, war, ye..."
20,lifestyle_Topic_2_Probability,female_seconds,-0.3305,9.628772e-30,1112,transcripts_lifestyle,"T2: आज, आपक, लग, अच, चल, कर, हम, रह, बन, इतन"
21,lifestyle_Topic_4_Probability,male_seconds,-0.3244,1.170689e-28,1112,transcripts_lifestyle,"T4: like, going, really, one, also, know, thin..."
9,religion_Topic_7_Probability,male_seconds,0.3183,3.219996e-22,882,transcripts_religion,"T7: angels, angel, angelology, three, soil, we..."
18,lifestyle_Topic_1_Probability,female_seconds,0.3045,2.731719e-25,1112,transcripts_lifestyle,"T1: skin, shade, foundation, brush, concealer,..."
12,religion_Topic_9_Probability,male_seconds,-0.2938,5.154296e-19,882,transcripts_religion,"T9: मन, आपक, कर, हम, उसक, अच, रह, अगर, कह, भगव"
13,religion_Topic_9_Probability,female_seconds,0.2803,2.188088e-17,882,transcripts_religion,"T9: मन, आपक, कर, हम, उसक, अच, रह, अगर, कह, भगव"


Cell 4 — Extract Tm/Tw word lists per community (rule-based, not hand-picked)

In [ ]:
DWEAT_INPUT_DIR = f'{BASE}/dweat_inputs'
os.makedirs(DWEAT_INPUT_DIR, exist_ok=True)

for community, short_name in COMMUNITY_SHORT_NAME.items():
    comm_sig = topic_results_df[topic_results_df['community'] == community]
    if comm_sig.empty:
        print(f"{community}: no significant topic-gender correlations, skipping D-WEAT input")
        continue

    male_rows = comm_sig[(comm_sig['gender'] == 'male_seconds') & (comm_sig['correlation'] > 0)]
    female_rows = comm_sig[(comm_sig['gender'] == 'female_seconds') & (comm_sig['correlation'] > 0)]

    def extract_words(rows):
        words = []
        for _, row in rows.iterrows():
            raw = row.get('topic_words', '')
            if isinstance(raw, str) and ':' in raw:
                words.extend(w.strip() for w in raw.split(':', 1)[1].split(','))
        seen, out = set(), []
        for w in words:
            if w and w not in seen:
                seen.add(w); out.append(w)
        return out

    male_words = extract_words(male_rows)
    female_words = extract_words(female_rows)

    with open(f'{DWEAT_INPUT_DIR}/topic_male_{short_name}.txt', 'w', encoding='utf-8') as f:
        f.write(" ".join(male_words))
    with open(f'{DWEAT_INPUT_DIR}/topic_female_{short_name}.txt', 'w', encoding='utf-8') as f:
        f.write(" ".join(female_words))

    print(f"{community}: male_words={len(male_words)}, female_words={len(female_words)}")

transcripts_business: male_words=17, female_words=10
transcripts_religion: male_words=29, female_words=10
transcripts_comedy: male_words=10, female_words=0
transcripts_lifestyle: male_words=10, female_words=20
transcripts_tech: male_words=43, female_words=10
politics_transcripts: no significant topic-gender correlations, skipping D-WEAT input
gaming_transcripts: no significant topic-gender correlations, skipping D-WEAT input
motivational_transcripts: no significant topic-gender correlations, skipping D-WEAT input
